# `examples/basic` 基础仿真示例中文注释版

本 Notebook 对应 `examples/basic/main.py`，用于演示 faas-sim 中最基础、最完整的一条仿真链路：

1. 创建边缘网络拓扑；
2. 初始化 Docker 镜像仓库；
3. 注册函数镜像；
4. 构造函数、函数镜像、函数容器和函数部署对象；
5. 将函数部署到 FaaS 系统；
6. 等待函数副本进入可调用状态；
7. 发起函数调用请求；
8. 推进 SimPy 离散事件仿真流程。

该示例适合作为后续调试 faas-sim 的起点，也适合作为理解 `Topology -> Benchmark -> Simulation -> Environment -> FaaS System` 主链路的最小样例。

## 1. 导入依赖

这一部分导入 faas-sim 基础示例所需的模块。

需要注意的是，Notebook 的当前工作目录可能不是项目根目录。因此下面会先尝试把项目根目录加入 `sys.path`，避免在 Jupyter 中出现 `ModuleNotFoundError: No module named 'sim'` 或 `No module named 'examples'`。

In [1]:
# 标准库：日志输出、路径处理和类型标注。
import logging
import sys
from pathlib import Path
from typing import List

# Notebook 运行目录兼容处理：
# 如果当前 Notebook 放在 examples/basic/ 下运行，则 PROJECT_ROOT 取两级父目录；
# 如果当前 Notebook 放在项目根目录运行，则 PROJECT_ROOT 就是当前目录。
current_dir = Path.cwd()
candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
]

for root in candidate_roots:
    if (root / "sim").exists() and (root / "examples").exists():
        PROJECT_ROOT = root
        break
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"当前工作目录：{current_dir}")
print(f"推断项目根目录：{PROJECT_ROOT}")

# Ether：用于生成边缘/云网络拓扑。
# UrbanSensingScenario 是官方示例中常用的城市感知场景拓扑。
import ether.scenarios.urbansensing as scenario

# Skippy 工具函数：把 '58M'、'1024Mi' 这类可读容量字符串转换为字节数。
from skippy.core.utils import parse_size_string

# faas-sim Docker 镜像仓库与镜像属性。
from sim import docker
from sim.docker import ImageProperties

# Benchmark 是 faas-sim 实验场景的基类，用户需要实现 setup 和 run 两个阶段。
from sim.benchmark import Benchmark

# Environment 是仿真运行期上下文，内部包含拓扑、FaaS 系统、容器仓库、指标记录器等对象。
from sim.core import Environment

# faas-sim FaaS 抽象：
# Function：函数逻辑抽象；
# FunctionImage：函数镜像抽象；
# FunctionContainer：函数容器运行配置；
# FunctionDeployment：函数部署对象；
# FunctionRequest：函数调用请求；
# ScalingConfiguration：伸缩配置；
# DeploymentRanking：多镜像部署优先级；
# KubernetesResourceConfiguration：Kubernetes 风格资源请求配置。
from sim.faas import (
    FunctionDeployment,
    FunctionRequest,
    Function,
    FunctionImage,
    ScalingConfiguration,
    DeploymentRanking,
    FunctionContainer,
    KubernetesResourceConfiguration,
)

# Simulation 是 faas-sim 的顶层入口，负责组装拓扑、Benchmark 和运行环境。
from sim.faassim import Simulation

# faas-sim 拓扑类型。
from sim.topology import Topology

# 模块级日志记录器，用于输出示例运行过程中的部署、请求和调试信息。
logger = logging.getLogger(__name__)

当前工作目录：c:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\basic
推断项目根目录：c:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master


## 2. 创建示例拓扑

`example_topology()` 负责构造基础实验使用的拓扑对象。

在这个示例中，拓扑来自 Ether 的 `UrbanSensingScenario`。它会生成一个城市感知场景中的节点、链路和网络结构，用于模拟边缘环境中的计算与网络资源。

最后调用 `t.init_docker_registry()` 初始化 Docker 镜像仓库节点，使后续函数部署时可以模拟镜像拉取过程。

In [2]:
def example_topology() -> Topology:
    """
    创建基础示例使用的网络拓扑。

    业务含义：
    - Topology 是 faas-sim 对计算节点、网络节点、链路和连接关系的统一封装；
    - Ether 的 UrbanSensingScenario 会向拓扑中填充一组预设的城市感知边缘节点；
    - init_docker_registry 会在拓扑中初始化容器镜像仓库，用于后续模拟镜像分发和拉取。

    返回：
    - Topology：已经完成节点、链路、场景和镜像仓库初始化的拓扑对象。
    """
    # 创建空拓扑。此时拓扑中还没有节点和网络连接。
    t = Topology()

    # 将城市感知场景实例化到拓扑中。
    # 这一步会向拓扑添加计算节点、网络链路和场景相关标签。
    scenario.UrbanSensingScenario().materialize(t)

    # 初始化 Docker 镜像仓库。
    # 函数副本部署到节点时，如果目标节点没有对应镜像，就需要从该仓库模拟拉取镜像。
    t.init_docker_registry()

    return t

## 3. 定义基础 Benchmark

在 faas-sim 中，`Benchmark` 表示一次仿真实验的“场景脚本”。

一个 Benchmark 通常由两个核心阶段组成：

- `setup(env)`：在仿真正式运行前准备环境，例如注册镜像、初始化存储索引、配置后台监控器等；
- `run(env)`：定义仿真时间推进过程，例如部署函数、等待副本可用、生成请求、等待请求完成等。

下面的 `ExampleBenchmark` 会部署两个函数：

- `python-pi`：模拟普通 CPU 函数；
- `resnet50-inference`：模拟 ResNet50 推理函数，同时提供 CPU 和 GPU 两类镜像。

In [3]:
class ExampleBenchmark(Benchmark):
    """
    基础实验场景。

    业务职责：
    1. 在 setup 阶段向容器镜像仓库注册函数镜像；
    2. 在 run 阶段创建函数部署对象并调用 env.faas.deploy 部署函数；
    3. 等待函数副本进入可用状态；
    4. 构造 FunctionRequest 并通过 env.faas.invoke 发起函数请求；
    5. 等待所有请求执行完成，使 Metrics 中形成完整的部署、调度、调用和网络记录。

    该类展示了 faas-sim 最基础的实验组织方式，是理解后续自定义函数仿真器、
    自定义调度器、自动伸缩器和结果分析的入口。
    """

    def setup(self, env: Environment):
        """
        准备仿真实验所需的容器镜像。

        参数：
        - env：faas-sim 运行时环境，内部持有 container_registry、faas、metrics、topology 等核心对象。

        业务流程：
        1. 取得环境中的容器镜像仓库 env.container_registry；
        2. 注册 python-pi-cpu 在 arm32、x86、aarch64 三种架构下的镜像；
        3. 注册 resnet50-inference-cpu 在三种架构下的镜像；
        4. 注册 resnet50-inference-gpu 在三种架构下的镜像；
        5. 打印镜像仓库内容，便于确认镜像是否已经正确注册。

        说明：
        - 这里的镜像注册不是拉取真实 Docker 镜像，而是向仿真环境声明“镜像存在”；
        - 镜像大小会影响后续部署阶段的网络传输时间；
        - 架构字段会影响调度阶段对节点可运行性的判断。
        """
        # 取得 faas-sim 内部的容器镜像仓库。
        containers: docker.ContainerRegistry = env.container_registry

        # 注册 python-pi 函数的 CPU 镜像。
        # 同一个逻辑镜像名可以对应多个 CPU 架构，调度时会根据节点架构选择可用镜像。
        containers.put(ImageProperties("python-pi-cpu", parse_size_string("58M"), arch="arm32"))
        containers.put(ImageProperties("python-pi-cpu", parse_size_string("58M"), arch="x86"))
        containers.put(ImageProperties("python-pi-cpu", parse_size_string("58M"), arch="aarch64"))

        # 注册 resnet50 推理函数的 CPU 镜像。
        containers.put(ImageProperties("resnet50-inference-cpu", parse_size_string("56M"), arch="arm32"))
        containers.put(ImageProperties("resnet50-inference-cpu", parse_size_string("56M"), arch="x86"))
        containers.put(ImageProperties("resnet50-inference-cpu", parse_size_string("56M"), arch="aarch64"))

        # 注册 resnet50 推理函数的 GPU 镜像。
        # 该镜像用于表达“同一个函数可以有不同实现版本”，例如 CPU 版本和 GPU 版本。
        containers.put(ImageProperties("resnet50-inference-gpu", parse_size_string("56M"), arch="arm32"))
        containers.put(ImageProperties("resnet50-inference-gpu", parse_size_string("56M"), arch="x86"))
        containers.put(ImageProperties("resnet50-inference-gpu", parse_size_string("56M"), arch="aarch64"))

        # 输出镜像仓库当前内容。
        # containers.images 的结构通常是：镜像名 -> tag -> ImageProperties 列表。
        for name, tag_dict in containers.images.items():
            for tag, images in tag_dict.items():
                logger.info("镜像名称=%s, tag=%s, 镜像属性=%s", name, tag, images)

    def run(self, env: Environment):
        """
        执行基础仿真实验主流程。

        参数：
        - env：faas-sim 运行时环境。

        业务流程：
        1. 构造函数部署对象；
        2. 逐个部署函数；
        3. 等待每个函数至少有一个可用副本；
        4. 并发发起 10 个 python-pi 请求；
        5. 并发发起 10 个 resnet50-inference 请求；
        6. 等待所有请求完成。

        产出：
        - 该方法本身不返回普通 Python 值，而是作为 SimPy 协程运行；
        - 通过 yield / yield from 把控制权交给 SimPy 事件调度器；
        - 实验结果写入 env.metrics，可在仿真结束后提取 DataFrame 分析。
        """
        # 准备两个函数部署对象：python-pi 和 resnet50-inference。
        deployments = self.prepare_deployments()

        # 部署所有函数。
        # env.faas.deploy 是一个 SimPy 事件过程，因此需要 yield from 等待其部署流程完成。
        for deployment in deployments:
            yield from env.faas.deploy(deployment)

        # 等待函数副本进入可用状态。
        # poll_available_replica 会持续检查函数是否已有 running 副本。
        logger.info("等待 python-pi 和 resnet50-inference 的函数副本进入可调用状态")
        yield env.process(env.faas.poll_available_replica("python-pi"))
        yield env.process(env.faas.poll_available_replica("resnet50-inference"))

        # ps 用于保存并发请求对应的 SimPy Process。
        # 后续逐个 yield，确保所有请求都执行完成。
        ps = []

        logger.info("发起 10 个 python-pi 请求")
        for i in range(10):
            # FunctionRequest 只指定函数名，FaaS 系统内部会通过负载均衡器选择具体副本。
            ps.append(env.process(env.faas.invoke(FunctionRequest("python-pi"))))

        logger.info("发起 10 个 resnet50-inference 请求")
        for i in range(10):
            ps.append(env.process(env.faas.invoke(FunctionRequest("resnet50-inference"))))

        # 等待所有请求完成。
        # 如果不等待，仿真可能在请求尚未完成时结束，导致指标不完整。
        for p in ps:
            yield p

    def prepare_deployments(self) -> List[FunctionDeployment]:
        """
        创建本实验需要部署的函数列表。

        返回：
        - List[FunctionDeployment]：函数部署对象列表。

        说明：
        - FunctionDeployment 是 faas-sim 中函数上线的核心对象；
        - 它同时包含函数定义、容器实现、伸缩配置和部署优先级。
        """
        resnet_fd = self.prepare_resnet_inference_deployment()
        python_pi_fd = self.prepare_python_pi_deployment()

        # 返回顺序决定 run 中部署函数的顺序。
        return [python_pi_fd, resnet_fd]

    def prepare_python_pi_deployment(self) -> FunctionDeployment:
        """
        创建 python-pi 函数部署对象。

        业务含义：
        - python-pi 表示一个普通 CPU 函数；
        - 它只有一个函数镜像 python-pi-cpu；
        - 它没有显式资源请求配置，因此使用默认资源配置；
        - 它使用默认 ScalingConfiguration，表示按 faas-sim 默认伸缩配置部署。

        返回：
        - FunctionDeployment：python-pi 的部署对象。
        """
        # 函数逻辑名称。调用 FunctionRequest 时使用该名称。
        python_pi = "python-pi"

        # 函数镜像。FunctionImage 表示函数的一种实现方式。
        python_pi_cpu = FunctionImage(image="python-pi-cpu")

        # Function 是函数的设计时抽象，包含函数名称和候选镜像实现。
        python_pi_fn = Function(python_pi, fn_images=[python_pi_cpu])

        # FunctionContainer 是运行时容器配置，绑定一个 FunctionImage。
        # 这里没有额外设置资源请求，表示使用默认资源配置。
        python_pi_fn_container = FunctionContainer(python_pi_cpu)

        # FunctionDeployment 是部署时对象。
        # 它将函数定义、可部署容器和伸缩配置组合起来，供 FaaS 系统部署。
        python_pi_fd = FunctionDeployment(
            python_pi_fn,
            [python_pi_fn_container],
            ScalingConfiguration(),
        )

        return python_pi_fd

    def prepare_resnet_inference_deployment(self) -> FunctionDeployment:
        """
        创建 resnet50-inference 函数部署对象。

        业务含义：
        - resnet50-inference 表示一个 AI 推理函数；
        - 它同时具有 CPU 镜像和 GPU 镜像；
        - GPU 容器设置了更高的内存资源请求；
        - DeploymentRanking 指定优先使用 GPU 镜像，其次使用 CPU 镜像。

        返回：
        - FunctionDeployment：resnet50-inference 的部署对象。
        """
        # 函数逻辑名称。
        resnet_inference = "resnet50-inference"

        # 两种镜像实现：CPU 推理版本和 GPU 推理版本。
        inference_cpu = "resnet50-inference-cpu"
        inference_gpu = "resnet50-inference-gpu"

        # 构造函数镜像对象。
        resnet_inference_gpu = FunctionImage(image=inference_gpu)
        resnet_inference_cpu = FunctionImage(image=inference_cpu)

        # Function 中同时注册 GPU 和 CPU 两种实现，表示该函数可以被部署为不同执行形态。
        resnet_fn = Function(
            resnet_inference,
            fn_images=[resnet_inference_gpu, resnet_inference_cpu],
        )

        # CPU 容器配置：使用默认资源请求。
        resnet_cpu_container = FunctionContainer(resnet_inference_cpu)

        # GPU 容器配置：声明 Kubernetes 风格资源请求。
        # cpu='100m' 表示请求 0.1 个 CPU；
        # memory='1024Mi' 表示请求 1024 MiB 内存。
        request = KubernetesResourceConfiguration.create_from_str(cpu="100m", memory="1024Mi")
        resnet_gpu_container = FunctionContainer(resnet_inference_gpu, resource_config=request)

        # DeploymentRanking 表示部署候选镜像的优先级。
        # 这里优先尝试 GPU 镜像，如果调度条件不满足，再考虑 CPU 镜像。
        resnet_fd = FunctionDeployment(
            resnet_fn,
            [resnet_cpu_container, resnet_gpu_container],
            ScalingConfiguration(),
            DeploymentRanking([inference_gpu, inference_cpu]),
        )

        return resnet_fd

## 4. 运行基础仿真

下面的 `main()` 函数与 `examples/basic/main.py` 的入口一致。

运行后会完成：

- 构造拓扑；
- 构造 Benchmark；
- 创建 Simulation；
- 调用 `sim.run()` 推进仿真；
- 输出部分关键指标表，便于在 Notebook 中直接观察结果。

In [4]:
def main():
    """
    基础示例入口函数。

    业务流程：
    1. 初始化日志级别；
    2. 创建拓扑；
    3. 创建基础 Benchmark；
    4. 创建 Simulation；
    5. 运行仿真；
    6. 返回 Simulation 对象，便于 Notebook 后续继续分析 metrics。
    """
    logging.basicConfig(level=logging.INFO)

    # 创建拓扑。此处会生成 UrbanSensingScenario 并初始化镜像仓库。
    topology = example_topology()

    # 创建实验场景。Benchmark 会在 Simulation.run() 中被调用。
    benchmark = ExampleBenchmark()

    # 创建仿真对象。Simulation 会组装 Environment、FaaS System、网络和指标对象。
    sim = Simulation(topology, benchmark)

    # 启动仿真。内部会先执行 benchmark.setup(env)，再执行 benchmark.run(env)。
    sim.run()

    return sim


# 执行基础仿真。
sim = main()

INFO:sim.faassim:initializing simulation, benchmark: ExampleBenchmark, topology nodes: 144
INFO:sim.faassim:starting resource monitor
INFO:sim.faassim:setting up benchmark
INFO:__main__:镜像名称=python-pi-cpu, tag=latest, 镜像属性=[ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='arm32'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='x86'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='aarch64')]
INFO:__main__:镜像名称=resnet50-inference-cpu, tag=latest, 镜像属性=[ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='arm32'), ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='x86'), ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='aarch64')]
INFO:__main__:镜像名称=resnet50-inference-gpu, tag=latest, 镜像属性=[ImageProperties(name='resnet50-inference-gpu', size=56000000, tag='latest', arch='arm32'), ImageProperties(name='resnet50-inference-gpu

## 5. 查看仿真指标

faas-sim 会将部署、调度、调用、网络流、资源使用等事件写入 `sim.env.metrics`。

下面提取若干常见 DataFrame，观察基础示例是否运行成功。

In [5]:
# 从 Metrics 中提取函数调用记录。
# invocations_df 通常包含函数名、开始时间、结束时间、执行耗时等字段。
invocations_df = sim.env.metrics.extract_dataframe("invocations")

# 从 Metrics 中提取调度记录。
# schedule_df 通常包含函数副本、候选节点、最终节点、镜像等调度相关信息。
schedule_df = sim.env.metrics.extract_dataframe("schedule")

# 从 Metrics 中提取副本部署记录。
# replica_deployment_df 可用于观察每个函数副本的部署阶段耗时。
replica_deployment_df = sim.env.metrics.extract_dataframe("replica_deployment")

# 从 Metrics 中提取网络流记录。
# flow_df 可用于观察镜像拉取或数据传输过程中的网络流。
flow_df = sim.env.metrics.extract_dataframe("flow")

print("函数调用记录行数：", len(invocations_df))
print("调度记录行数：", len(schedule_df))
print("副本部署记录行数：", len(replica_deployment_df))
print("网络流记录行数：", len(flow_df))

函数调用记录行数： 20
调度记录行数： 6
副本部署记录行数： 8
网络流记录行数： 2


In [6]:
# 查看函数调用结果。
# 如果基础示例运行正常，这里应该至少能看到 python-pi 和 resnet50-inference 的调用记录。
invocations_df.head()

,t_wait,t_exec,t_start,memory,function_name,function_image,node,replica_id
time,,,,,,,,
2026-07-03 18:11:11.516935,0.0,1.0,1.0,1048576,python-pi,python-pi-cpu,server_0,1278279618240
2026-07-03 18:11:11.516957,0.0,1.0,1.0,1048576,python-pi,python-pi-cpu,server_0,1278279618240
2026-07-03 18:11:11.516967,0.0,1.0,1.0,1048576,python-pi,python-pi-cpu,server_0,1278279618240
2026-07-03 18:11:11.516975,0.0,1.0,1.0,1048576,python-pi,python-pi-cpu,server_0,1278279618240
2026-07-03 18:11:11.516983,0.0,1.0,1.0,1048576,python-pi,python-pi-cpu,server_0,1278279618240


In [7]:
# 计算平均函数执行时间。
# t_exec 是 faas-sim 中常用的函数执行耗时字段；如果当前版本字段名不同，可先查看 invocations_df.columns。
if "t_exec" in invocations_df.columns:
    print("平均函数执行时间：", invocations_df["t_exec"].mean())
else:
    print("当前 invocations_df 字段：")
    print(invocations_df.columns)

平均函数执行时间： 1.0


In [9]:
# 查看调度结果。
# 这有助于确认函数副本被调度到了哪些节点，以及使用了哪些镜像。
schedule_df.head()

,value,function_name,image,replica_id,node_name,successful
time,,,,,,
2026-07-03 18:11:11.487484,queue,python-pi,python-pi-cpu,1278279618240,NaN,NaN
2026-07-03 18:11:11.488270,queue,resnet50-inference,resnet50-inference-gpu,1278279100176,NaN,NaN
2026-07-03 18:11:11.488319,start,python-pi,python-pi-cpu,1278279618240,NaN,NaN
2026-07-03 18:11:11.503332,finish,python-pi,python-pi-cpu,1278279618240,server_0,True
2026-07-03 18:11:11.505016,start,resnet50-inference,resnet50-inference-gpu,1278279100176,NaN,NaN


## 6. 调试建议

如果运行过程中报错，可以优先检查：

1. 当前 Notebook 是否位于项目根目录或能正确推断项目根目录；
2. 是否已经执行 `pip install -e .`；
3. `requirements.txt` 中外部依赖是否安装完整；
4. 内置的 `ether`、`skippy`、`simpy` 是否位于项目根目录；
5. `srds` 是否已经通过 pip 安装；
6. 如果 Metrics DataFrame 为空，检查是否所有请求都被 `yield` 等待完成。